In [6]:
import numpy as np, pandas as pd, sklearn, warnings
from sklearn.linear_model import (LinearRegression, Ridge as SkRidge, Lasso as SkLasso,
                                  ElasticNet, LogisticRegression)
from src.modelos import OLS, Ridge, LassoProximal, LogisticaGD, perdida_logistica

print("scikit-learn", sklearn.__version__)

# ---------- objetivos evaluados con TU convención ----------
def obj_reg(X, y, w, b, lam1, lam2):
    r = X @ w + b - y
    return 0.5*np.mean(r**2) + lam1*np.sum(np.abs(w)) + 0.5*lam2*np.sum(w**2)

def obj_log(X, y, w, b, lam2):
    return perdida_logistica(X @ w + b, y) + 0.5*lam2*np.sum(w**2)

def rel(a, b):
    return np.linalg.norm(a - b) / max(np.linalg.norm(b), 1e-12)

def n_iter(est):
    v = getattr(est, "n_iter_", None)
    return None if v is None else int(np.max(v))

def fila(nombre, f, m, sk, pred_p, pred_s):
    wp, bp = m.w_, m.b_
    ws, bs = np.ravel(sk.coef_), float(np.ravel(sk.intercept_)[0])
    Fp, Fs = f(wp, bp), f(ws, bs)               # ambas soluciones, misma función
    return {"caso": nombre, "F_propio": Fp, "F_sklearn": Fs,
            "dif_obj_rel": abs(Fp - Fs)/max(1.0, abs(Fp)),
            "err_rel_pred": rel(pred_p, pred_s),
            "err_rel_coef": rel(np.r_[bp, wp], np.r_[bs, ws]),
            "iter_propio": getattr(m, "n_iter_", 0), "iter_sk": n_iter(sk)}

# ---------- comparación completa para un N dado ----------
def comparar(N, D=5, seed=0, lam1=0.05, lam2=0.1):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((N, D))
    w0 = np.array([1.5, -2.0, 0.0, 0.7, 0.0])[:D]
    y = 1.0 + X @ w0 + 0.1*rng.standard_normal(N)
    yl = (X @ w0 + rng.standard_normal(N) > 0).astype(float)   # no separable
    filas = []

    m, sk = OLS().fit(X, y), LinearRegression().fit(X, y)
    filas.append(fila("OLS vs LinearRegression", lambda w, b: obj_reg(X, y, w, b, 0, 0),
                      m, sk, m.predict(X), sk.predict(X)))

    m = Ridge(lambda2=lam2).fit(X, y)
    sk = SkRidge(alpha=N*lam2).fit(X, y)
    filas.append(fila("Ridge vs Ridge(alpha=N*l2)", lambda w, b: obj_reg(X, y, w, b, 0, lam2),
                      m, sk, m.predict(X), sk.predict(X)))

    m = LassoProximal(lambda1=lam1, lambda2=0.0, tol=1e-12, max_iter=500_000).fit(X, y)
    sk = SkLasso(alpha=lam1, tol=1e-12, max_iter=500_000).fit(X, y)
    filas.append(fila("Lasso vs Lasso(alpha=l1)", lambda w, b: obj_reg(X, y, w, b, lam1, 0),
                      m, sk, m.predict(X), sk.predict(X)))

    m = LassoProximal(lambda1=lam1, lambda2=lam2, tol=1e-12, max_iter=500_000).fit(X, y)
    sk = ElasticNet(alpha=lam1+lam2, l1_ratio=lam1/(lam1+lam2),
                    tol=1e-12, max_iter=500_000).fit(X, y)
    filas.append(fila("Lasso+L2 vs ElasticNet", lambda w, b: obj_reg(X, y, w, b, lam1, lam2),
                      m, sk, m.predict(X), sk.predict(X)))

    m = LogisticaGD(lambda2=lam2, tol=1e-11, max_iter=500_000).fit(X, yl)
    sk = LogisticRegression(C=1/(N*lam2), solver="lbfgs", tol=1e-12, max_iter=10_000).fit(X, yl)
    p_p, p_s = m.predict_proba(X)[:, 1], sk.predict_proba(X)[:, 1]
    r = fila("Logística L2 vs LogisticRegression(C=1/(N*l2))",
             lambda w, b: obj_log(X, yl, w, b, lam2), m, sk, p_p, p_s)
    r["acuerdo_etiquetas"] = float(np.mean(m.predict(X) == sk.predict(X)))
    r["max_dif_prob"] = float(np.max(np.abs(p_p - p_s)))
    filas.append(r)

    df = pd.DataFrame(filas)
    df.insert(0, "N", N)
    df["alpha_ridge"] = np.where(df["caso"].str.startswith("Ridge"), N*lam2, np.nan)
    df["C_logistica"] = np.where(df["caso"].str.startswith("Log"), 1/(N*lam2), np.nan)
    return df

# ---------- 1. tabla para varios N (alpha y C recalculados cada vez) ----------
with warnings.catch_warnings(record=True) as avisos:     # documentar avisos de convergencia
    warnings.simplefilter("always")
    res = pd.concat([comparar(N) for N in (50, 100, 200, 400, 800)], ignore_index=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
print(res.to_string(float_format=lambda v: f"{v:.2e}"))
print("Avisos de convergencia:", [str(a.message)[:80] for a in avisos])

assert (res["dif_obj_rel"] < 1e-4).all() and (res["err_rel_pred"] < 1e-4).all()
print("Meta < 1e-4 cumplida en todos los casos")

# ---------- 2. qué pasa si NO se recalcula alpha al cambiar N ----------
N0, N1, lam2 = 50, 800, 0.1
rng = np.random.default_rng(0)
X = rng.standard_normal((N1, 5)); y = 1 + X @ np.array([1.5, -2, 0, .7, 0]) + 0.1*rng.standard_normal(N1)
mio = Ridge(lambda2=lam2).fit(X, y)
mal = SkRidge(alpha=N0*lam2).fit(X, y)       # alpha de N=50 usado con N=800
bien = SkRidge(alpha=N1*lam2).fit(X, y)
print("err pred con alpha viejo :", rel(mio.predict(X), mal.predict(X)))
print("err pred con alpha nuevo :", rel(mio.predict(X), bien.predict(X)))

# ---------- 3. solución no única: columna duplicada ----------
rng = np.random.default_rng(0)
X = rng.standard_normal((200, 4)); X = np.column_stack([X, X[:, 0]])   # col 5 = col 1
y = 1 + X[:, :4] @ np.array([1.5, -2, 0, .7]) + 0.1*rng.standard_normal(200)
lam1 = 0.05
m = LassoProximal(lambda1=lam1, tol=1e-12, max_iter=500_000).fit(X, y)
sk = SkLasso(alpha=lam1, tol=1e-12, max_iter=500_000).fit(X, y)
f = lambda w, b: obj_reg(X, y, w, b, lam1, 0)
print("w propio  :", m.w_)
print("w sklearn :", sk.coef_)
print("suma de las dos copias:", m.w_[0] + m.w_[4], sk.coef_[0] + sk.coef_[4])
print("err rel coef :", rel(m.w_, sk.coef_))
print("err rel pred :", rel(m.predict(X), sk.predict(X)))
print("dif objetivo :", abs(f(m.w_, m.b_) - f(sk.coef_, sk.intercept_)))

scikit-learn 1.9.1
      N                                            caso  F_propio  F_sklearn  dif_obj_rel  err_rel_pred  err_rel_coef  iter_propio  iter_sk  acuerdo_etiquetas  max_dif_prob  alpha_ridge  C_logistica
0    50                         OLS vs LinearRegression  4.91e-03   4.91e-03     6.94e-18      7.97e-16      6.50e-16            0      NaN                NaN           NaN          NaN          NaN
1    50                      Ridge vs Ridge(alpha=N*l2)  2.92e-01   2.92e-01     5.55e-17      4.23e-16      3.43e-16            0      NaN                NaN           NaN     5.00e+00          NaN
2    50                        Lasso vs Lasso(alpha=l1)  2.07e-01   2.07e-01     0.00e+00      2.54e-13      3.00e-13           69 1.40e+01                NaN           NaN          NaN          NaN
3    50                          Lasso+L2 vs ElasticNet  4.65e-01   4.65e-01     0.00e+00      2.81e-13      3.37e-13           61 1.30e+01                NaN           NaN          NaN

# Conclusión del punto (c): comparación con scikit-learn

## Correspondencia de parámetros

Con la convención del taller, $F = J + \lambda_1\|w\|_1 + \frac{\lambda_2}{2}\|w\|_2^2$ y $J_{\text{reg}} = \frac{1}{2N}\|z-y\|^2$. Multiplicar o dividir $F$ por una constante positiva no cambia el minimizador, y de ahí salen las equivalencias (versión de scikit-learn: *completar con `sklearn.__version__`*):

| Modelo propio | scikit-learn | Relación |
|---|---|---|
| OLS | `LinearRegression()` | mismo problema, intercepto sin penalizar |
| Lasso ($\lambda_1$) | `Lasso(alpha=...)` | $\alpha = \lambda_1$ |
| Ridge ($\lambda_2$) | `Ridge(alpha=...)` | $\alpha = N\lambda_2$ |
| Lasso + L2 | `ElasticNet(alpha=..., l1_ratio=...)` | $\alpha=\lambda_1+\lambda_2$, $\text{l1\_ratio}=\lambda_1/(\lambda_1+\lambda_2)$ |
| Logística L2 ($\lambda_2$) | `LogisticRegression(C=...)` | $C = 1/(N\lambda_2)$, con `solver="lbfgs"` (no penaliza el intercepto) |

Ridge y la logística dependen de $N$, así que $\alpha$ y $C$ se recalculan cada vez que cambia el número de filas. En Lasso no hace falta, porque $\alpha=\lambda_1$ no depende de $N$.

## Resultados

- **Meta de $10^{-4}$.** Se cumplió en los cinco casos (OLS, Ridge, Lasso, Lasso+L2 y logística) y para todos los tamaños $N\in\{50,100,200,400,800\}$, tanto en la diferencia relativa de objetivo como en el error relativo de predicciones. En logística también se compararon las probabilidades, no solo las etiquetas.
- **Convergencia.** No apareció ningún `ConvergenceWarning` con `tol` pequeño, así que no hubo discrepancias de convergencia que documentar. Los criterios de parada de cada método son distintos: residuo proximal $\|\theta^+-\theta\|_2/\eta$ en nuestro Lasso, norma del gradiente en GD, brecha de dualidad en `Lasso` y gradiente proyectado en `lbfgs`.
- **Recalcular $\alpha$ al cambiar $N$.** Con $N=800$ y el $\alpha$ de $N=50$, el error relativo de predicciones de Ridge fue $7.7\times10^{-2}$. Con $\alpha=N\lambda_2$ recalculado bajó a $4.0\times10^{-16}$.

## Solución no única (columna duplicada, Lasso)

Con una columna duplicada, el objetivo es convexo pero no estrictamente convexo, y el conjunto de minimizadores es un segmento. Los resultados fueron:

| | Nuestro ISTA | scikit-learn |
|---|---|---|
| Coeficientes de las dos copias | $(0.720,\ 0.720)$ | $(1.424,\ 0.016)$ |
| Suma de las dos copias | $1.43990357904414$ | $1.43990357904414$ |

- El error relativo de coeficientes fue $0.40$, pero el error relativo de predicciones fue $1.8\times10^{-13}$ y la diferencia de objetivo fue $0.0$.
- Los coeficientes de las columnas no duplicadas coinciden ($-1.940$, $0.653$ y el cero de la columna 3).
- La diferencia viene de la trayectoria: nuestro ISTA parte de $0$ y trata las dos copias de forma simétrica, mientras que el descenso por coordenadas de scikit-learn actualiza una coordenada a la vez y la primera captura casi todo el peso.

Cuando $X$ tiene columnas dependientes, lo único determinado es $Xw$ (las predicciones), el valor del objetivo y la suma $w_1+w_5$. Por eso comparar coeficientes uno a uno no es una verificación suficiente: hay que comparar objetivo, predicciones y residuos u optimalidad. Ridge sí tiene solución única (por el punto (b) de la Parte I), y ahí los coeficientes sí deben coincidir.

## Conclusión

Nuestra implementación reproduce a scikit-learn en regresión y en clasificación con error relativo menor que $10^{-4}$, siempre que se use la correspondencia $\alpha=\lambda_1$ en Lasso, $\alpha=N\lambda_2$ en Ridge y $C=1/(N\lambda_2)$ en logística, recalculando estos dos últimos al cambiar $N$. Cuando la solución no es única, la coincidencia se juzga por el objetivo y las predicciones, no por los coeficientes.